# Finishing the Indic voice pipeline — Colab T4

Runs the remaining measurements and records them as files. Read
`docs/AGENT_BRIEF.md` in the repo for what each phase decides and the
decision rules (fixed in advance, so a disappointing number cannot be
reinterpreted afterwards).

**Before running:** Runtime → Change runtime type → **T4 GPU**. Add `HF_TOKEN`
in the Secrets panel (🔑) with notebook access enabled.

Order matters. Phase 1 is a regression check on ASR decode guards whose effect
was never measured; if it fails, everything after it is measuring a broken
baseline.

| Phase | Decides | Time |
| --- | --- | --- |
| 1 | are the decode guards a regression? | ~20 min |
| 2 | LLM, TTS backend, streaming grid, CT2 engine, compiled decode | ~2 h |
| 3 | real conversational latency, as a distribution | ~15 min, needs your microphone |

Phases 1–2 are unattended and **resumable** — if the session dies, re-run the
same cell and finished steps are skipped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## 1. Clone and install

`fetch && reset --hard` rather than `pull`: local installs and result files
make `pull` refuse, and that refusal is easy to miss.

In [ ]:
%cd /content
!rm -rf indic-voice-pipeline
!git clone -q https://github.com/Vaibhav7711/indic-voice-pipeline.git
%cd /content/indic-voice-pipeline
!git log --oneline -1
!pip install -q -e ".[dev,demo,audio]" faster-whisper ctranslate2 2>&1 | tail -2
!python scripts/preflight.py

### Gate: the test suite must pass

A failing suite invalidates every number after it. Expect `563 passed, 8 skipped`
(the 8 are GPU-gated and run inside the validation sweep).

In [ ]:
!python -m pytest -q -p no:cacheprovider 2>&1 | tail -3

## 2. Drive, Hugging Face, and the adapter

Drive is where evidence survives a dead session. The v2 adapter is not in git
— it comes from the private checkpoint repo the training run wrote to.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
MIRROR = '/content/drive/MyDrive/ivp-results'
os.makedirs(MIRROR, exist_ok=True)

from huggingface_hub import login, snapshot_download, whoami
login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
CKPT_REPO = f"{whoami()['name']}/whisper-turbo-hindi-lora-ckpt"
ADAPTER = snapshot_download(CKPT_REPO, allow_patterns=['best/*'],
                            local_dir='/content/v2-final') + '/best'
print('adapter:', ADAPTER, sorted(os.listdir(ADAPTER)))
assert os.path.exists(f'{ADAPTER}/adapter_model.safetensors'), 'adapter weights missing'

## 3. Phase 1 — decode-guard regression check

Two evaluations one flag apart. The guards (no-speech threshold 0.6,
repetition loop guard) were added without measuring their effect on WER. A
no-speech suppression on a clip that *does* contain speech deletes a whole
utterance.

In [ ]:
!python scripts/bench_all.py --adapter {ADAPTER} --mirror {MIRROR} \
    --only guards_on,guards_off,compare_guards 2>&1 | tail -20

### Read Phase 1 before continuing

**Keep the defaults** if WER is unchanged or better and the guards fired on
0–2 clips. **Treat it as a regression to fix** if WER is worse, or if
`no_speech` fired on a clip whose reference is non-empty — raise
`--no-speech-threshold` and re-run with `--force-steps guards_on`.

In [ ]:
import json, pathlib

def load(path):
    p = pathlib.Path(path)
    return json.loads(p.read_text()) if p.is_file() else None

for name in ('v2-guards-off', 'v2-guards-on'):
    m = load(f'results/eval/{name}/metrics.json')
    if not m:
        print(f'{name}: MISSING — check results/bench_logs/'); continue
    g = m.get('decode_guards_fired', {})
    print(f"{name:14s} WER {m['headline']['wer_percent']:6.2f}%  "
          f"CER {m['headline']['cer_percent']:5.2f}%  "
          f"no_speech fired {g.get('no_speech', 0)}  repetition {g.get('repetition', 0)}")
    if g.get('no_speech_ids'):
        print('   suppressed:', g['no_speech_ids'])

cmp_ = load('results/eval/compare-guards.json')
print('\ncomparison:', json.dumps(cmp_, indent=1)[:900] if cmp_ else 'MISSING')

### If clips were suppressed, check whether they actually contain speech

A suppressed clip with a non-empty reference is a deleted utterance, not a
saved one.

In [ ]:
rows = [json.loads(l) for l in open('results/eval/v2-guards-on/predictions.jsonl',
                                    encoding='utf-8')] \
       if pathlib.Path('results/eval/v2-guards-on/predictions.jsonl').is_file() else []
suppressed = [r for r in rows if r.get('no_speech')]
print(f'{len(suppressed)} clip(s) suppressed by the no-speech check')
for r in suppressed[:10]:
    print(f"\n{r['id']}  P(nospeech)={r.get('no_speech_prob')}")
    print('  reference:', r['reference'][:120])
    print('  -> reference is', 'EMPTY (correct suppression)' if not r['reference'].strip()
          else 'NON-EMPTY — this is a deleted utterance, raise the threshold')

## 4. Phase 2 — the open questions (~2 h, unattended)

LLM choice, TTS backend, streaming session grid, CTranslate2 engine, compiled
decode. All three LLM candidates fit a 16 GB T4 unquantized, so this runs in
fp16.

If the session dies, re-run this same cell — finished steps are skipped.

In [ ]:
!python scripts/bench_all.py --adapter {ADAPTER} --mirror {MIRROR} \
    --llm-models Qwen/Qwen3-0.6B,Qwen/Qwen3-1.7B,Qwen/Qwen3-4B \
    --skip guards_on,guards_off,compare_guards 2>&1 | tail -30

### What ran, what failed

In [ ]:
man = load('results/bench_manifest.json')
for s in (man or {}).get('steps', []):
    status = 'skipped' if s.get('skipped') else ('ok' if s['returncode'] == 0 else 'FAILED')
    print(f"{s['step']:14s} {status:8s} {s['seconds']/60:5.1f} min  {s['decides']}")
    if s['returncode'] and not s.get('skipped'):
        print('   ', s['tail'][:200])

### LLM bake-off — the table ranks latency, **you** rank Hindi

Apply the rule: highest `devanagari_ratio_mean` with **zero** `think_leaks`
and `first_sentence_ms_p50` < 800 ms. Then read the answers yourself — a
model that scores well and still restates the question is not the winner.

In [ ]:
s = load('results/llm_bakeoff/summary.json')
for name, st in (s or {}).get('models', {}).items():
    if 'error' in st:
        print(f'{name:24s} ERROR {st["error"][:80]}'); continue
    print(f"{name:24s} {st['params_millions']:5d}M  VRAM {st['peak_vram_gib']:4.1f}  "
          f"prefill {st['prefill_ms_mean']:6.0f}  {st['ms_per_token_mean']:5.1f} ms/tok  "
          f"1st-sent p50 {st['first_sentence_ms_p50']:6.0f}  "
          f"deva {st['devanagari_ratio_mean']}  think {st['think_leaks']}  "
          f"rep {st['repetition_stops']}")

In [ ]:
import glob
for f in sorted(glob.glob('results/llm_bakeoff/*.outputs.jsonl')):
    print('\n' + '=' * 70)
    print(f.split('/')[-1].replace('.outputs.jsonl', ''))
    for row in list(map(json.loads, open(f, encoding='utf-8')))[:6]:
        print(f"\nQ: {row['prompt']}\nA: {row.get('response', row.get('error'))}")

### TTS backend

In [ ]:
s = load('results/tts_bakeoff/summary.json')
for name, r in (s or {}).get('backends', {}).items():
    if 'error' in r:
        print(f'{name:8s} ERROR {r["error"][:80]}'); continue
    print(f"{name:8s} streaming={r['streaming']!s:5s} first-chunk p50 "
          f"{r['first_chunk_ms_p50']:6.0f} ms  RTF {r['rtf_mean']:.2f}")
print('\nlisten before choosing:')
import IPython.display as ipd
for w in sorted(glob.glob('results/tts_bakeoff/*.wav'))[:4]:
    print(w); ipd.display(ipd.Audio(w))

### Streaming session grid

Rules: adopt `early-incr` if `wer_vs_offline` rises < 1.0 pp versus `early`
**and** mean `endpoint_to_final_ms` drops. Adopt `early-sem` if split clips
fall from 16/100 to single digits at < 150 ms mean added latency.

In [ ]:
s = load('results/streaming_eval/summary.json')
print('offline WER vs reference:', (s or {}).get('offline_wer_vs_reference'))
print(f"\n{'config':14s} {'WER ref':>8s} {'WER off':>8s} {'split':>6s} {'empty':>6s} "
      f"{'onset':>6s} {'end→final':>10s} {'asr after':>10s}")
for name, r in (s or {}).get('configs', {}).items():
    print(f"{name:14s} {r['wer_vs_reference']:8.2f} {r['wer_vs_offline']:8.2f} "
          f"{r['clips_split']:6d} {r['clips_empty']:6d} {r['onset_hallucinations']:6d} "
          f"{(r.get('endpoint_to_final_ms') or 0):10.0f} "
          f"{(r.get('asr_after_endpoint_ms') or 0):10.0f}")

### Validation sweep, and the engine tier

`ct2_matches_explicit` must pass before any CTranslate2 speedup counts;
`llm_compiled_matches_eager` measured 0.9× (slower) on a T4 before.

In [ ]:
for d in ('results/gpu_validation', 'results/gpu_validation-ct2'):
    rep = load(f'{d}/report.json')
    if not rep:
        print(f'{d}: MISSING'); continue
    print('\n' + d)
    for c in rep['checks']:
        keys = {k: v for k, v in c['detail'].items()
                if isinstance(v, (int, float, bool)) and k != 'traceback'}
        print(f"  [{c['status']:4s}] {c['name']:32s} "
              f"{ {k: (round(v, 3) if isinstance(v, float) else v) for k, v in list(keys.items())[:4]} }")
        if c['status'] == 'fail':
            print('        ', (c['error'] or '')[:180])

## 5. Phase 3 — live turns (needs your microphone)

Twelve or more turns, varied lengths, including follow-ups that exercise
dialogue memory (e.g. ask about Delhi, then "और मुंबई का?"). Allow microphone
access when the browser asks.

Set `LLM` to the Phase 2 winner before running.

In [ ]:
LLM = 'Qwen/Qwen3-1.7B'      # <- the Phase 2 winner
TTS = 'mms'                   # <- 'edge' or 'mms', per the TTS result

from demo.notebook import NotebookAgent, audio_report, record
agent = NotebookAgent(adapter=ADAPTER, llm=LLM, tts=TTS).load()

In [ ]:
clip = record(6)
audio_report(clip)
agent.stream_turn(clip)

Re-run the cell above for each turn. When you have ≥ 12, save and summarise
— the deliverable is a **distribution**, not one example.

In [ ]:
import numpy as np, os
os.makedirs('results/live', exist_ok=True)
with open('results/live/turns.jsonl', 'w', encoding='utf-8') as fh:
    for t in agent.turns:
        fh.write(json.dumps(t, ensure_ascii=False) + '\n')

lat = [t['response_latency_ms'] for t in agent.turns if t.get('response_latency_ms')]
print(f'{len(agent.turns)} turns, {len(lat)} with a measured response latency')
if lat:
    print(f'  response latency  p50 {np.percentile(lat,50):6.0f} ms   '
          f'p90 {np.percentile(lat,90):6.0f} ms   mean {np.mean(lat):6.0f} ms')
for key in ('asr_ms', 'first_token_ms', 'to_audio_ms',
            'first_token_to_first_unit_ms', 'tts_synthesis_ms'):
    vals = [t[key] for t in agent.turns if t.get(key) is not None]
    if vals:
        print(f'  {key:30s} mean {np.mean(vals):6.0f} ms  (n={len(vals)})')
print('\nsummary():', agent.summary())

## 6. Keep the evidence

Mirrored to Drive as each step finished. This also zips it for download, so it
can be committed from a machine with push access.

In [ ]:
!cp -r results/live {MIRROR}/ 2>/dev/null; true
!cd /content/indic-voice-pipeline && zip -qr /content/ivp-evidence.zip \
    results/bench_manifest.json results/bench_logs results/eval results/llm_bakeoff \
    results/tts_bakeoff results/streaming_eval results/gpu_validation \
    results/gpu_validation-ct2 results/live -x "*.wav" "*.mp3" "*.bin" && \
    ls -la /content/ivp-evidence.zip
from google.colab import files; files.download('/content/ivp-evidence.zip')

## 7. Write the handover

`results/SESSION.md` is for a reader who was not here: what was decided, what
changed, what is still open. Edit the text below before writing it — do not
record a decision the numbers above do not support.

In [ ]:
session = '''# Session summary

## Decided
- Decode guards: <keep / fixed, with the threshold used>
- LLM: <winner, and why; note if pending human listening>
- TTS backend: <winner, and why>
- Incremental finals: <on/off, per the rule>
- Semantic endpointing: <on/off, per the rule>
- CTranslate2 engine: <adopted/not, speedup, token-match result>
- Compiled decode: <on/off, measured speedup>

## Measured
- Live turns: <n> turns, response latency p50 <x> ms / p90 <y> ms
- Evidence: results/ (mirrored to Drive)

## Still open
- data/hard_set/ is empty — needs human clip curation
- Device-level barge-in unmeasurable in Colab; laptop-validated only

## For the next session
- <what to pick up>
'''
open('results/SESSION.md', 'w').write(session)
print(session)